# Proposed WESAD model: Cross-Participant Generalization and Adaptive Personalization (CGAP)

This faster notebook evaluates CGAP with **three random seeds on one fixed participant-held-out split**. It compares CGAP with Majority Dummy, Logistic Regression, and RBF SVM. The held-out participant is never used for model fitting or hyperparameter selection.

CGAP combines a shared ExtraTrees classifier with an SGD classifier that is updated online. Each held-out window is predicted first; its true label is then used to update the online branch. This is a **labeled-feedback replay protocol**, not an unlabeled real-time deployment. Results from one held-out participant must not be reported as 15-subject LOSO performance.

The tuning grids and inner validation repetitions are reduced to make the notebook run faster. All results are checkpointed to Google Drive after each seed.


## 1. Connect Drive and load the corrected participant CSVs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
subject_files = sorted(p for p in DATA_DIR.glob('S*_windows.csv') if re.fullmatch(r'S\d+_windows', p.stem))
if len(subject_files) != 15:
    raise FileNotFoundError(f'Expected 15 corrected participant files in {DATA_DIR}; found {len(subject_files)}')
df = pd.concat((pd.read_csv(path) for path in subject_files), ignore_index=True)
print('Files:', len(subject_files), '| rows:', len(df), '| participants:', df.subject.nunique())
display(df.head())

## 2. Validate the inputs

Participant and window identifiers are used only for splitting and ordering. Labels and label IDs are never model features.

In [ ]:
required = {'subject', 'window', 'label', 'label_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
FEATURES = [column for column in df.columns if column not in required]
assert FEATURES, 'No sensor feature columns found'
df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='coerce')
assert df[FEATURES].notna().all().all(), 'Missing feature values: check that only corrected participant CSVs were loaded'
assert np.isfinite(df[FEATURES].to_numpy(dtype=float)).all(), 'Infinite feature values found'
assert df.duplicated().sum() == 0, 'Duplicate rows found'
assert df.groupby('subject').window.apply(lambda values: values.is_monotonic_increasing).all(), 'Sort participant rows by window before proceeding'
CLASSES = ['baseline', 'stress', 'amusement']
assert set(df.label.astype(str).unique()) == set(CLASSES), f'Unexpected labels: {sorted(df.label.astype(str).unique())}'
assert df.subject.nunique() == 15, f'Expected 15 participants; got {df.subject.nunique()}'
df = df.reset_index(drop=True)
X = df[FEATURES].astype(float)
y = df.label.astype(str)
groups = df.subject.astype(str)
print('Rows:', len(df), '| features:', len(FEATURES), '| classes:', y.value_counts().to_dict())

## 3. Define fast tuning and evaluation

A single participant is held out for final testing. The remaining participants are used for inner group-based tuning and training. Tuning grids are deliberately compact for runtime. This is a quick experiment, not a substitute for full nested LOSO evaluation.


In [ ]:
from itertools import product
import copy
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
import json
import time

SEEDS = [17, 42, 2026]

# Compact grids for a quicker run.
LR_GRID = list(product([0.1, 1.0, 10.0], ['balanced']))
SVM_GRID = list(product([1.0, 10.0], ['scale', 0.1], ['balanced']))
TREE_GRID = [(0.8, 2), (1.0, 5)]
SGD_ALPHA_GRID = [0.0001, 0.001]
BLEND_GRID = [0.25, 0.75]
INNER_SPLITS = 2
CHUNK_SIZE = 10

metric_columns = ['accuracy', 'balanced_accuracy', 'macro_f1', 'macro_precision', 'macro_recall']

def score(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
    }

def make_batch_model(name, params, seed):
    if name == 'Logistic Regression':
        c_value, class_weight = params
        classifier = LogisticRegression(C=c_value, class_weight=class_weight, max_iter=1500, random_state=seed)
    elif name == 'RBF SVM':
        c_value, gamma, class_weight = params
        classifier = SVC(C=c_value, gamma=gamma, kernel='rbf', class_weight=class_weight)
    else:
        raise ValueError(name)
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), classifier)

def make_cgap_global(params, seed):
    max_features, min_leaf = params
    return make_pipeline(
        SimpleImputer(strategy='median'),
        ExtraTreesClassifier(
            n_estimators=80, max_features=max_features,
            min_samples_leaf=min_leaf, class_weight='balanced',
            n_jobs=-1, random_state=seed
        )
    )

def new_online_model(alpha, seed):
    return SGDClassifier(loss='log_loss', alpha=alpha, class_weight=None,
                         max_iter=1000, tol=1e-3, random_state=seed)

def ordered_positions(frame):
    return frame.sort_values(['subject', 'window']).index.to_numpy()

def tune_batch(name, train_positions, seed):
    grid = LR_GRID if name == 'Logistic Regression' else SVM_GRID
    inner = GroupShuffleSplit(n_splits=INNER_SPLITS, test_size=0.25, random_state=seed)
    x_train = X.loc[train_positions]
    y_train = y.loc[train_positions]
    g_train = groups.loc[train_positions]
    evaluated = []
    for params in grid:
        fold_scores = []
        for fit_local, val_local in inner.split(x_train, y_train, groups=g_train):
            model = make_batch_model(name, params, seed)
            model.fit(x_train.iloc[fit_local], y_train.iloc[fit_local])
            pred = model.predict(x_train.iloc[val_local])
            fold_scores.append(f1_score(y_train.iloc[val_local], pred, labels=CLASSES, average='macro', zero_division=0))
        evaluated.append((float(np.mean(fold_scores)), params))
    return max(evaluated, key=lambda item: item[0])

def fit_cgap(train_positions, tree_params, sgd_alpha, seed):
    train_x, train_y = X.loc[train_positions], y.loc[train_positions]
    global_model = make_cgap_global(tree_params, seed)
    global_model.fit(train_x, train_y)

    scaler = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
    train_scaled = scaler.fit_transform(train_x)
    online_model = new_online_model(sgd_alpha, seed)

    class_counts = train_y.value_counts()
    missing = set(CLASSES) - set(class_counts.index)
    if missing:
        raise ValueError(f'Training split is missing classes: {missing}')
    class_weight_by_label = {
        label: len(train_y) / (len(CLASSES) * class_counts[label])
        for label in CLASSES
    }
    fit_weights = np.asarray([class_weight_by_label[label] for label in train_y])
    online_model.fit(train_scaled, train_y, sample_weight=fit_weights)
    return global_model, scaler, online_model, class_weight_by_label

def evaluate_cgap(global_model, scaler, online_model, class_weight_by_label, blend, frame, chunk_size=10):
    positions = ordered_positions(frame)
    actual, predicted, curve = [], [], []
    for count, position in enumerate(positions, start=1):
        raw_x = X.loc[[position]]
        online_x = scaler.transform(raw_x)
        true_label = y.loc[position]

        p_online_local = online_model.predict_proba(online_x)[0]
        p_online = np.array([p_online_local[list(online_model.classes_).index(label)] for label in CLASSES])
        p_global_local = global_model.predict_proba(raw_x)[0]
        p_global = np.array([p_global_local[list(global_model.classes_).index(label)] for label in CLASSES])

        probabilities = (1.0 - blend) * p_global + blend * p_online
        pred = CLASSES[int(np.argmax(probabilities))]
        actual.append(true_label)
        predicted.append(pred)

        # Predict first, then update using the true label (labeled-feedback protocol).
        online_model.partial_fit(
            online_x, [true_label], classes=np.asarray(CLASSES),
            sample_weight=np.asarray([class_weight_by_label[true_label]])
        )
        if count % chunk_size == 0 or count == len(positions):
            curve.append({'windows_seen': count, **score(actual, predicted)})
    return actual, predicted, curve

def tune_cgap(train_positions, seed):
    train_frame = df.loc[train_positions]
    inner = GroupShuffleSplit(n_splits=INNER_SPLITS, test_size=0.25, random_state=seed)
    scored = []
    for tree_params in TREE_GRID:
        for sgd_alpha in SGD_ALPHA_GRID:
            for blend in BLEND_GRID:
                fold_scores = []
                for fit_local, val_local in inner.split(train_frame, groups=train_frame.subject.astype(str)):
                    fit_positions = train_frame.iloc[fit_local].index.to_numpy()
                    val_positions = train_frame.iloc[val_local].index.to_numpy()
                    global_model, scaler, online_model, class_weights = fit_cgap(
                        fit_positions, tree_params, sgd_alpha, seed
                    )
                    # Score each validation participant independently. Start each
                    # participant from the same trained online state so adaptation
                    # does not carry across validation participants.
                    val_frame = df.loc[val_positions]
                    subject_scores = []
                    for val_subject in sorted(val_frame.subject.astype(str).unique()):
                        subject_frame = val_frame[val_frame.subject.astype(str) == val_subject]
                        online_copy = copy.deepcopy(online_model)
                        actual, predicted, _ = evaluate_cgap(
                            global_model, scaler, online_copy, class_weights,
                            blend, subject_frame, chunk_size=10**9
                        )
                        subject_scores.append(
                            f1_score(actual, predicted, labels=CLASSES,
                                     average='macro', zero_division=0)
                        )
                    fold_scores.append(float(np.mean(subject_scores)))
                scored.append((float(np.mean(fold_scores)), tree_params, sgd_alpha, blend))
    return max(scored, key=lambda item: item[0])


## 4. Run three seeds on one fixed participant-held-out split

Choose one participant as the final test subject. All other participants are used for training and inner tuning. Each seed repeats tuning and fitting, then saves results immediately. This is faster than 15-fold LOSO but only evaluates the selected participant.


In [ ]:
from pathlib import Path
import json
import pandas as pd

# Change this value if you want a different fixed test participant.
HOLDOUT_SUBJECT = sorted(groups.unique())[-1]
test_mask = groups.astype(str).eq(str(HOLDOUT_SUBJECT)).to_numpy()
test_pos = np.flatnonzero(test_mask)
train_pos = np.flatnonzero(~test_mask)

train_subjects = set(groups.iloc[train_pos])
test_subjects = set(groups.iloc[test_pos])
assert len(test_subjects) == 1 and train_subjects.isdisjoint(test_subjects)

test_frame = df.iloc[test_pos].copy()
print('Held-out subject:', HOLDOUT_SUBJECT)
print('Training subjects:', len(train_subjects), '| train windows:', len(train_pos), '| test windows:', len(test_pos))

CHECKPOINT_DIR = DATA_DIR / 'cgap_fast_checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = CHECKPOINT_DIR / 'results_checkpoint.csv'
SELECTED_FILE = CHECKPOINT_DIR / 'selected_checkpoint.csv'
CURVE_FILE = CHECKPOINT_DIR / 'curve_checkpoint.csv'

results = pd.read_csv(RESULTS_FILE) if RESULTS_FILE.exists() else pd.DataFrame()
selected_parameters = pd.read_csv(SELECTED_FILE) if SELECTED_FILE.exists() else pd.DataFrame()
learning_curve = pd.read_csv(CURVE_FILE) if CURVE_FILE.exists() else pd.DataFrame()

# This checkpoint folder is dedicated to the seed-only protocol.
completed = set()
if not results.empty and {'seed', 'model'}.issubset(results.columns):
    completed = set(results.loc[results.model.eq('CGAP'), 'seed'].astype(int))
print('Completed seeds:', sorted(completed))

def save_checkpoint():
    results.to_csv(RESULTS_FILE, index=False)
    selected_parameters.to_csv(SELECTED_FILE, index=False)
    learning_curve.to_csv(CURVE_FILE, index=False)
    print('Checkpoint saved:', CHECKPOINT_DIR, flush=True)

for seed in SEEDS:
    seed = int(seed)
    if seed in completed:
        print(f'Skipping completed seed={seed}', flush=True)
        continue

    started = time.time()
    print(f'\nSeed {seed}', flush=True)
    seed_records, seed_selected, seed_curve = [], [], []

    # Majority baseline
    dummy = make_pipeline(SimpleImputer(strategy='median'), DummyClassifier(strategy='prior'))
    dummy.fit(X.iloc[train_pos], y.iloc[train_pos])
    dummy_pred = dummy.predict(X.iloc[test_pos])
    seed_records.append({
        'seed': seed, 'held_out_subject': HOLDOUT_SUBJECT, 'model': 'Majority Dummy',
        'best_inner_macro_f1': np.nan, 'best_params': '{}',
        **score(y.iloc[test_pos], dummy_pred)
    })

    # Tuned batch baselines
    for model_name in ['Logistic Regression', 'RBF SVM']:
        inner_score, params = tune_batch(model_name, train_pos, seed)
        model = make_batch_model(model_name, params, seed)
        model.fit(X.iloc[train_pos], y.iloc[train_pos])
        prediction = model.predict(X.iloc[test_pos])
        seed_records.append({
            'seed': seed, 'held_out_subject': HOLDOUT_SUBJECT, 'model': model_name,
            'best_inner_macro_f1': inner_score, 'best_params': json.dumps(params),
            **score(y.iloc[test_pos], prediction)
        })
        seed_selected.append({
            'seed': seed, 'held_out_subject': HOLDOUT_SUBJECT, 'model': model_name,
            'inner_macro_f1': inner_score, 'best_params': json.dumps(params)
        })

    # CGAP
    inner_score, tree_params, sgd_alpha, blend = tune_cgap(train_pos, seed)
    global_model, scaler, online_model, class_weights = fit_cgap(
        train_pos, tree_params, sgd_alpha, seed
    )
    actual, prediction, curve = evaluate_cgap(
        global_model, scaler, online_model, class_weights,
        blend, test_frame, chunk_size=CHUNK_SIZE
    )
    params = {
        'tree_max_features': tree_params[0],
        'tree_min_samples_leaf': tree_params[1],
        'sgd_alpha': sgd_alpha,
        'online_blend_weight': blend
    }
    seed_records.append({
        'seed': seed, 'held_out_subject': HOLDOUT_SUBJECT, 'model': 'CGAP',
        'best_inner_macro_f1': inner_score, 'best_params': json.dumps(params),
        **score(actual, prediction)
    })
    seed_selected.append({
        'seed': seed, 'held_out_subject': HOLDOUT_SUBJECT, 'model': 'CGAP',
        'inner_macro_f1': inner_score, 'best_params': json.dumps(params)
    })
    for point in curve:
        seed_curve.append({
            'seed': seed, 'held_out_subject': HOLDOUT_SUBJECT, **point
        })

    results = pd.concat([results, pd.DataFrame(seed_records)], ignore_index=True)
    selected_parameters = pd.concat([selected_parameters, pd.DataFrame(seed_selected)], ignore_index=True)
    learning_curve = pd.concat([learning_curve, pd.DataFrame(seed_curve)], ignore_index=True)
    save_checkpoint()
    completed.add(seed)

    cgap_row = next(row for row in seed_records if row['model'] == 'CGAP')
    print(
        f"Finished seed={seed}; CGAP macro-F1={cgap_row['macro_f1']:.3f}; "
        f"elapsed={time.time()-started:.1f}s",
        flush=True
    )

print('Result records:', len(results), '| expected when all seeds complete:', 4 * len(SEEDS))
display(results.tail(12))


## 5. Summaries and participant-held-out learning curve

Average metrics across the three seeds for the selected held-out participant. Because there is only one held-out participant, the standard deviation here reflects seed variability only, not variability across participants.


## 6. Optional comparison with previously reported HAT results

Previously reported values are descriptive references from a different run. Do not claim a fair head-to-head comparison unless the earlier model is rerun using the same held-out participant, data preprocessing, feedback protocol, and evaluation metrics.


In [ ]:
# Mean metrics across seeds for each model on the fixed held-out participant
per_subject = results.groupby(['model', 'held_out_subject'], as_index=False)[metric_columns].mean()

previous_macro_f1 = {
    'Hoeffding Adaptive Tree (ADWIN), previous run': 0.560,
    'Logistic Regression, previous run': 0.457,
    'Majority Dummy, previous run': 0.232,
    'RBF SVM, previous run': 0.491,
}
comparison_rows = [
    {'model': model, 'macro_f1_mean': value, 'source': 'previous run; not directly comparable'}
    for model, value in previous_macro_f1.items()
]
for model in results.model.unique():
    value = float(per_subject.loc[per_subject.model == model, 'macro_f1'].mean())
    comparison_rows.append({
        'model': model, 'macro_f1_mean': value,
        'source': 'this notebook; fixed held-out participant'
    })
comparison = pd.DataFrame(comparison_rows)
display(comparison.sort_values('macro_f1_mean', ascending=False).reset_index(drop=True))


In [ ]:
# Metric summary across seeds
metric_columns = ['accuracy', 'balanced_accuracy', 'macro_f1', 'macro_precision', 'macro_recall']
per_subject = results.groupby(['model', 'held_out_subject'], as_index=False)[metric_columns].mean()
summary = results.groupby('model')[metric_columns].agg(['mean', 'std', 'min', 'max']).round(3)
display(summary)

print('Selected parameters by model and frequency:')
display(
    selected_parameters.groupby(['model', 'best_params'])
    .size().rename('seed_count').reset_index()
    .sort_values(['model', 'seed_count'], ascending=[True, False])
)

print('CGAP cumulative metrics by windows seen:')
if not learning_curve.empty:
    display(learning_curve.groupby('windows_seen')[['accuracy', 'balanced_accuracy', 'macro_f1']].mean().round(3))


## 7. Save reproducible outputs to Drive

The notebook saves per-seed scores, selected hyperparameters, summaries, and CGAP's chronological learning curve. The output is explicitly labeled as a single-participant holdout experiment.


In [ ]:
OUTPUT_DIR = DATA_DIR / 'cgap_fast_results'
OUTPUT_DIR.mkdir(exist_ok=True)

results.to_csv(OUTPUT_DIR / 'cgap_fixed_holdout_3seeds_raw.csv', index=False)
per_subject.to_csv(OUTPUT_DIR / 'cgap_fixed_holdout_per_subject.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'cgap_fixed_holdout_summary.csv')
selected_parameters.to_csv(OUTPUT_DIR / 'cgap_selected_hyperparameters.csv', index=False)
learning_curve.to_csv(OUTPUT_DIR / 'cgap_online_learning_curve.csv', index=False)
comparison.to_csv(OUTPUT_DIR / 'cgap_comparison_reference.csv', index=False)

print('Saved results to:', OUTPUT_DIR)


## How to interpret results

A higher CGAP macro-F1 than the baselines on this fixed held-out participant is preliminary evidence for this labeled-feedback replay setting. Three seeds on one participant do **not** establish generalization across all 15 WESAD participants. For publication-level performance claims, run the full participant-held-out LOSO experiment and report per-participant results and uncertainty.

**Runtime choices:** compact tuning grids, two inner group splits, 80 ExtraTrees estimators, and one fixed test participant. The notebook still performs tuning for each seed, so runtime depends on dataset size and Colab hardware.


## 8. Publication-style metrics tables and diagnostic diagrams

This section adds reproducible figures for the **fixed held-out participant** using the selected hyperparameters from each seed. In this notebook, the held-out participant is the test subject; it is not an independent validation set. The tuning validation scores are used only for model selection and are not stored window-by-window. The code below re-fits each seed's selected model and records probability predictions for diagnostic plots.

It produces: overall metric comparison, confusion matrices, per-class precision/recall/F1, one-vs-rest ROC and precision–recall curves, calibration/reliability plots, confidence histogram, and CGAP online learning curves. These are exploratory single-participant figures; Q1-level generalization claims require all 15 outer LOSO folds.


In [ ]:
# Publication-style diagnostic figures and prediction exports
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    log_loss, accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score
)
from sklearn.calibration import calibration_curve

DIAG_DIR = DATA_DIR / "cgap_fast_diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)
CLASS_ORDER = list(CLASSES)

# Recover the chosen hyperparameters saved by the main experiment.
# The checkpoint may contain rows from previous runs, so retain this notebook's seeds.
chosen = selected_parameters[selected_parameters["seed"].astype(int).isin(SEEDS)].copy()
chosen = chosen.drop_duplicates(subset=["seed", "model"], keep="last")

prediction_rows = []
metric_rows = []
per_class_rows = []
cgap_curves_for_plot = []

def get_selected(seed, model_name):
    row = chosen[(chosen.seed.astype(int) == int(seed)) & (chosen.model == model_name)]
    if row.empty:
        raise ValueError(f"Missing selected parameters for seed={seed}, model={model_name}. Run the main experiment first.")
    return json.loads(row.iloc[-1]["best_params"])

def ordered_proba(model, x, class_order=CLASS_ORDER):
    raw = model.predict_proba(x)
    model_classes = list(model.classes_) if hasattr(model, "classes_") else list(model[-1].classes_)
    return np.column_stack([raw[:, model_classes.index(c)] for c in class_order])

for seed in SEEDS:
    seed = int(seed)
    # Batch baselines: refit on the same training participants with selected parameters.
    for model_name in ["Logistic Regression", "RBF SVM"]:
        params = tuple(get_selected(seed, model_name))
        model = make_batch_model(model_name, params, seed)
        model.fit(X.iloc[train_pos], y.iloc[train_pos])
        p = ordered_proba(model, X.iloc[test_pos])
        pred = np.asarray(CLASS_ORDER)[np.argmax(p, axis=1)]
        for j, pos in enumerate(test_pos):
            row = {
                "seed": seed, "subject": str(groups.iloc[pos]), "window": df.iloc[pos]["window"],
                "model": model_name, "y_true": str(y.iloc[pos]), "y_pred": str(pred[j]),
                "p_baseline": p[j, 0], "p_stress": p[j, 1], "p_amusement": p[j, 2]
            }
            prediction_rows.append(row)

    # Majority dummy baseline has no useful class probabilities for all classes in some folds,
    # but its predictions can still be included in the metric summary.
    dummy = make_pipeline(SimpleImputer(strategy="median"), DummyClassifier(strategy="prior"))
    dummy.fit(X.iloc[train_pos], y.iloc[train_pos])
    p_dummy_raw = dummy.predict_proba(X.iloc[test_pos])
    dummy_classes = list(dummy[-1].classes_)
    p_dummy = np.column_stack([
        p_dummy_raw[:, dummy_classes.index(c)] if c in dummy_classes else np.zeros(len(test_pos))
        for c in CLASS_ORDER
    ])
    p_dummy = np.clip(p_dummy, 1e-12, None)
    p_dummy = p_dummy / p_dummy.sum(axis=1, keepdims=True)
    pred_dummy = np.asarray(CLASS_ORDER)[np.argmax(p_dummy, axis=1)]
    for j, pos in enumerate(test_pos):
        prediction_rows.append({
            "seed": seed, "subject": str(groups.iloc[pos]), "window": df.iloc[pos]["window"],
            "model": "Majority Dummy", "y_true": str(y.iloc[pos]), "y_pred": str(pred_dummy[j]),
            "p_baseline": p_dummy[j, 0], "p_stress": p_dummy[j, 1], "p_amusement": p_dummy[j, 2]
        })

    # CGAP: predict first, then update from the true label, matching the notebook's labelled-feedback replay.
    params = get_selected(seed, "CGAP")
    tree_params = (params["tree_max_features"], int(params["tree_min_samples_leaf"]))
    sgd_alpha = float(params["sgd_alpha"])
    blend = float(params["online_blend_weight"])
    global_model, scaler, online_model, class_weights = fit_cgap(
        train_pos, tree_params, sgd_alpha, seed
    )
    positions = ordered_positions(test_frame)
    cgap_actual, cgap_pred, cgap_prob = [], [], []
    for count, pos in enumerate(positions, start=1):
        raw_x = X.loc[[pos]]
        online_x = scaler.transform(raw_x)
        p_online_local = online_model.predict_proba(online_x)[0]
        p_online = np.array([p_online_local[list(online_model.classes_).index(c)] for c in CLASS_ORDER])
        p_global_local = global_model.predict_proba(raw_x)[0]
        p_global = np.array([p_global_local[list(global_model.classes_).index(c)] for c in CLASS_ORDER])
        prob = (1.0 - blend) * p_global + blend * p_online
        pred = CLASS_ORDER[int(np.argmax(prob))]
        cgap_actual.append(str(y.loc[pos]))
        cgap_pred.append(pred)
        cgap_prob.append(prob)
        prediction_rows.append({
            "seed": seed, "subject": str(groups.loc[pos]), "window": df.loc[pos, "window"],
            "model": "CGAP", "y_true": str(y.loc[pos]), "y_pred": pred,
            "p_baseline": prob[0], "p_stress": prob[1], "p_amusement": prob[2]
        })
        online_model.partial_fit(
            online_x, [y.loc[pos]], classes=np.asarray(CLASS_ORDER),
            sample_weight=np.asarray([class_weights[y.loc[pos]]])
        )
        if count % CHUNK_SIZE == 0 or count == len(positions):
            cgap_curves_for_plot.append({
                "seed": seed, "windows_seen": count,
                "accuracy": accuracy_score(cgap_actual, cgap_pred),
                "balanced_accuracy": balanced_accuracy_score(cgap_actual, cgap_pred),
                "macro_f1": f1_score(cgap_actual, cgap_pred, labels=CLASS_ORDER, average="macro", zero_division=0)
            })

pred_df = pd.DataFrame(prediction_rows)
pred_path = DIAG_DIR / "heldout_window_predictions_with_probabilities.csv"
pred_df.to_csv(pred_path, index=False)

# Seed-wise metrics and per-class metrics.
for (seed, model_name), g in pred_df.groupby(["seed", "model"]):
    yt, yp = g.y_true.astype(str), g.y_pred.astype(str)
    metric_rows.append({
        "seed": int(seed), "model": model_name,
        "accuracy": accuracy_score(yt, yp),
        "balanced_accuracy": balanced_accuracy_score(yt, yp),
        "macro_precision": precision_score(yt, yp, labels=CLASS_ORDER, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, labels=CLASS_ORDER, average="macro", zero_division=0),
        "macro_f1": f1_score(yt, yp, labels=CLASS_ORDER, average="macro", zero_division=0)
    })
    report = classification_report(yt, yp, labels=CLASS_ORDER, output_dict=True, zero_division=0)
    for cls in CLASS_ORDER:
        per_class_rows.append({
            "seed": int(seed), "model": model_name, "class": cls,
            "precision": report[cls]["precision"], "recall": report[cls]["recall"],
            "f1": report[cls]["f1-score"], "support": report[cls]["support"]
        })

metric_df = pd.DataFrame(metric_rows)
class_df = pd.DataFrame(per_class_rows)
metric_df.to_csv(DIAG_DIR / "seedwise_validation_style_metrics.csv", index=False)
class_df.to_csv(DIAG_DIR / "per_class_metrics.csv", index=False)

# 1) Mean ± seed SD metric comparison.
metric_names = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1"]
agg = metric_df.groupby("model")[metric_names].agg(["mean", "std"])
agg.to_csv(DIAG_DIR / "metric_summary_mean_std.csv")
for metric in metric_names:
    means = metric_df.groupby("model")[metric].mean().sort_values(ascending=False)
    stds = metric_df.groupby("model")[metric].std().reindex(means.index).fillna(0)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(means.index, means.values, yerr=stds.values, capsize=4)
    ax.set_title(f"{metric.replace('_',' ').title()} by Model (mean ± seed SD)")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(DIAG_DIR / f"bar_{metric}.png", dpi=300, bbox_inches="tight")
    plt.show()

# 2) Confusion matrices: pooled windows across seeds (seeds are repeated runs, not independent participants).
for model_name in sorted(pred_df.model.unique()):
    g = pred_df[pred_df.model == model_name]
    cm = confusion_matrix(g.y_true, g.y_pred, labels=CLASS_ORDER, normalize=None)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Confusion Matrix — {model_name}\npooled across seeds")
    fig.tight_layout()
    fig.savefig(DIAG_DIR / f"confusion_matrix_{model_name.lower().replace(' ','_')}.png", dpi=300, bbox_inches="tight")
    plt.show()

    cm_norm = confusion_matrix(g.y_true, g.y_pred, labels=CLASS_ORDER, normalize="true")
    disp = ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_ORDER)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    disp.plot(ax=ax, cmap="Blues", colorbar=True, values_format=".2f")
    ax.set_title(f"Row-normalized Confusion Matrix — {model_name}")
    fig.tight_layout()
    fig.savefig(DIAG_DIR / f"confusion_matrix_normalized_{model_name.lower().replace(' ','_')}.png", dpi=300, bbox_inches="tight")
    plt.show()

# 3) Per-class precision, recall, F1 (averaged across seeds).
class_mean = class_df.groupby(["model", "class"])[["precision", "recall", "f1"]].mean().reset_index()
class_mean.to_csv(DIAG_DIR / "per_class_metrics_mean.csv", index=False)
for metric in ["precision", "recall", "f1"]:
    pivot = class_mean.pivot(index="class", columns="model", values=metric).reindex(CLASS_ORDER)
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"Per-class {metric.title()} (mean across seeds)")
    ax.set_ylabel(metric.title())
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Class")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(DIAG_DIR / f"per_class_{metric}.png", dpi=300, bbox_inches="tight")
    plt.show()

# 4) One-vs-rest ROC and precision-recall curves, pooled over seed predictions.
# Use these as descriptive plots only; repeated seed predictions are correlated.
prob_cols = ["p_baseline", "p_stress", "p_amusement"]
for model_name in sorted(pred_df.model.unique()):
    g = pred_df[pred_df.model == model_name]
    y_bin = np.column_stack([(g.y_true.values == cls).astype(int) for cls in CLASS_ORDER])
    probs = g[prob_cols].to_numpy()
    fig, ax = plt.subplots(figsize=(7, 6))
    for k, cls in enumerate(CLASS_ORDER):
        if len(np.unique(y_bin[:, k])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, k], probs[:, k])
        ax.plot(fpr, tpr, label=f"{cls} (AUC={auc(fpr,tpr):.3f})")
    ax.plot([0, 1], [0, 1], "--", label="Chance")
    ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title=f"One-vs-Rest ROC — {model_name}")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(DIAG_DIR / f"roc_{model_name.lower().replace(' ','_')}.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 6))
    for k, cls in enumerate(CLASS_ORDER):
        if len(np.unique(y_bin[:, k])) < 2:
            continue
        precision, recall, _ = precision_recall_curve(y_bin[:, k], probs[:, k])
        ap = average_precision_score(y_bin[:, k], probs[:, k])
        ax.plot(recall, precision, label=f"{cls} (AP={ap:.3f})")
    ax.set(xlabel="Recall", ylabel="Precision", title=f"One-vs-Rest Precision–Recall — {model_name}")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(DIAG_DIR / f"pr_{model_name.lower().replace(' ','_')}.png", dpi=300, bbox_inches="tight")
    plt.show()

# 5) Multiclass calibration/reliability and confidence distribution.
def multiclass_brier(y_true, probs, classes=CLASS_ORDER):
    y_onehot = np.column_stack([(np.asarray(y_true) == c).astype(float) for c in classes])
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))

calibration_rows = []
for model_name in sorted(pred_df.model.unique()):
    g = pred_df[pred_df.model == model_name]
    probs = g[prob_cols].to_numpy()
    true = g.y_true.to_numpy()
    conf = probs.max(axis=1)
    correct = (np.asarray(CLASS_ORDER)[probs.argmax(axis=1)] == true).astype(int)
    # Top-label reliability: confidence vs empirical accuracy in equal-width bins.
    bins = np.linspace(0, 1, 11)
    bin_conf, bin_acc, bin_n = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf >= lo) & ((conf < hi) if hi < 1 else (conf <= hi))
        if mask.any():
            bin_conf.append(float(conf[mask].mean()))
            bin_acc.append(float(correct[mask].mean()))
            bin_n.append(int(mask.sum()))
    brier = multiclass_brier(true, probs)
    try:
        ll = log_loss(true, probs, labels=CLASS_ORDER)
    except Exception:
        ll = np.nan
    calibration_rows.append({"model": model_name, "multiclass_brier": brier, "log_loss": ll})
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot([0, 1], [0, 1], "--", label="Perfect calibration")
    ax.plot(bin_conf, bin_acc, marker="o", label="Top-label reliability")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean confidence", ylabel="Accuracy",
           title=f"Reliability Diagram — {model_name}")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(DIAG_DIR / f"reliability_{model_name.lower().replace(' ','_')}.png", dpi=300, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(conf, bins=np.linspace(0, 1, 21), edgecolor="black")
    ax.set(xlabel="Maximum predicted class probability", ylabel="Number of windows",
           title=f"Prediction Confidence — {model_name}")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(DIAG_DIR / f"confidence_histogram_{model_name.lower().replace(' ','_')}.png", dpi=300, bbox_inches="tight")
    plt.show()

calibration_df = pd.DataFrame(calibration_rows)
calibration_df.to_csv(DIAG_DIR / "calibration_metrics.csv", index=False)

# 6) CGAP learning curve: mean ± seed SD over cumulative windows.
curve_df = pd.DataFrame(cgap_curves_for_plot)
curve_df.to_csv(DIAG_DIR / "cgap_learning_curve_diagnostic.csv", index=False)
if not curve_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
        grouped = curve_df.groupby("windows_seen")[metric]
        means, stds = grouped.mean(), grouped.std().fillna(0)
        ax.plot(means.index, means.values, marker="o", label=metric.replace("_", " ").title())
        ax.fill_between(means.index, np.maximum(0, means.values-stds.values),
                        np.minimum(1, means.values+stds.values), alpha=0.12)
    ax.set(xlabel="Held-out windows processed", ylabel="Metric",
           title="CGAP Online Replay Learning Curve (mean ± seed SD)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(DIAG_DIR / "cgap_online_learning_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

print("Saved all diagnostic CSVs and PNG diagrams to:", DIAG_DIR)
print("Window-level probability predictions:", pred_path)
display(metric_df.groupby("model")[metric_names].agg(["mean", "std"]).round(4))
display(class_mean.round(4))
display(calibration_df.round(4))


### 9. Inner-validation tuning score plots

The main experiment stores the **best inner-validation macro-F1** for each seed/model, but it does not store per-window inner-validation predictions. These plots therefore show tuning scores only; ROC, PR, confusion-matrix, and calibration plots above are generated on the fixed held-out participant.

In [ ]:
# Plot the validation score actually stored by the original tuning loop.
if "best_inner_macro_f1" in results.columns:
    val_scores = results.dropna(subset=["best_inner_macro_f1"]).copy()
    val_scores = val_scores[val_scores["seed"].astype(int).isin(SEEDS)]
    if not val_scores.empty:
        val_scores.to_csv(DIAG_DIR / "inner_validation_tuning_scores.csv", index=False)
        fig, ax = plt.subplots(figsize=(8, 5))
        for model_name, group in val_scores.groupby("model"):
            group = group.sort_values("seed")
            ax.plot(group["seed"].astype(str), group["best_inner_macro_f1"],
                    marker="o", linewidth=1.8, label=model_name)
        ax.set(title="Best Inner-Validation Macro-F1 by Seed",
               xlabel="Random seed", ylabel="Best inner-validation macro-F1")
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(DIAG_DIR / "inner_validation_macro_f1_by_seed.png", dpi=300, bbox_inches="tight")
        plt.show()

        summary_val = val_scores.groupby("model")["best_inner_macro_f1"].agg(["mean", "std", "min", "max"]).round(4)
        display(summary_val)
        summary_val.to_csv(DIAG_DIR / "inner_validation_macro_f1_summary.csv")
    else:
        print("No inner-validation scores found. Run the main experiment cell first.")
else:
    print("The results table has no best_inner_macro_f1 column; run the main experiment first.")